In [6]:
import os
import fastf1 as ff1
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
os.makedirs('fastf1-data', exist_ok=True)
ff1.Cache.enable_cache('fastf1-data')
years = [2018, 2019, 2021, 2022, 2023, 2024, 2025]
leclerc_monaco_history = []
for year in years:
  try:
    suzuka = ff1.get_session(year, 'Japan', 'R')
    suzuka.load(laps=True, telemetry=False, weather=False, messages=False)
    lec_suz = suzuka.results.loc[suzuka.results['Abbreviation'] == 'LEC']
    car_form = lec_suz['Position'].values[0] if not lec_suz.empty else 20.0

    session = ff1.get_session(year, 'Monaco', 'R')
    session.load(laps=True, telemetry=False, weather=False, messages=False)
    results = session.results
    leclerc = results.loc[results['Abbreviation'] == 'LEC']

    if not leclerc.empty:
          finish_pos = leclerc['Position'].values[0]
          grid_pos = leclerc['GridPosition'].values[0]

          leclerc_monaco_history.append({
              'Year': year,
              'Car_Form': car_form,
              'Grid_Position': grid_pos,
              'Finish_Position': finish_pos
            })
          print(f"{year} data secured!")

  except Exception as e:
      print(f"Oops, something went wrong in {year}: {e}")

df = pd.DataFrame(leclerc_monaco_history)
print(df)

In [ ]:
X_train = df[['Grid_Position', 'Car_Form']]
Y_train = df['Finish_Position']
model= RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)

In [55]:
leclerc_2026_prospects = pd.DataFrame([{
    'Grid_Position': 4.0,
    'Car_Form': 3.0
}])
prediction = model.predict(leclerc_2026_prospects)[0]
final_prediction = int(round(prediction))
final_prediction
print(f"Predicted Monaco Finish for Charles Leclerc: P{final_prediction}")

Predicted Monaco Finish for Charles Leclerc: P3
